# Sequential Feature Selection and Model Evaluation

In this notebook, a **comprehensive grid search** was performed across multiple feature selection strategies and classifiers to identify the most informative signal-derived and clinical features.

- **Overview**: 

    - Evaluated both **statistical** (SelectKBest) and **sequential** (SFS, RFE) feature selection methods
    - Included **correlation filtering** and tested various **signal preprocessing pipelines**
    - Compared a **diverse set of base estimators** (linear, tree-based, and neural)
    - Explored **multiple feature subset sizes and thresholds** to control redundancy

- **Search Space**:

    - **Preprocessing methods**: 5 (from direct interpolation to autoencoder-based gap filling)
    - **Feature selection methods**: SFS_forward, SFS_backward, RFE, and statistical filters
    - **Estimators**: Logistic Regression (L1/L2), Random Forest, Gradient Boosting, SVC, KNN, MLP (1- and 2-layer)
    - **Feature set sizes**: 4-15 features
    - **Correlation thresholds**: 0.5-0.9

- **Result Summary**:

Despite the extensive exploration, no model achieved satisfactory predictive power.

- The **best configuration** reached:
    - Mean CV accuracy: 0.697
    - Mean recall (or sensitivity): 0.338 P(y_hat​=1∣y=1)
    - Mean specificity: 0.892 P(y_hat​=0∣y=0)

- using:
    - Gradient Boosting
    - Backward Sequential Feature Selection
    - 15 features
    - extracted from drop-long-then-interpolate processed signals

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector, RFE, SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score

from helpers.feature_selection import *

from itertools import product

import warnings
import os
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [2]:
def run_sequential_search(
    base_features,
    preprocessing_methods, 
    sequential_methods, 
    estimators,
    n_features_options,
    corr_thresholds,
    cv_splits,
    stat_score_funcs
):

    """
    Run comprehensive sequential feature selection search
    """

    all_results = []
    
    # Part 1: Statistical methods (fast)
    print("\n" + "="*80)
    print("PART 1: STATISTICAL FEATURE SELECTION (with correlation filtering)")
    print("="*80)
    
    stat_param_grid = {
        'stage': [('stage_1', stage_1_df), ('stage_2', stage_2_df)],
        'preprocessing': preprocessing_methods,
        'score_func': list(stat_score_funcs.keys()),
        'estimator': list(estimators.keys()),
        'n_features': n_features_options,
        'corr_threshold': corr_thresholds
    }
    
    total_stat = np.prod([len(v) for v in stat_param_grid.values()])
    print(f"Statistical configurations to test: {total_stat}")
    
    iteration = 0
    
    for (stage_name, df), prep_method, score_name, est_name, n_feat, corr_thresh in product(
        stat_param_grid['stage'],
        stat_param_grid['preprocessing'],
        stat_param_grid['score_func'],
        stat_param_grid['estimator'],
        stat_param_grid['n_features'],
        stat_param_grid['corr_threshold']
    ):
        iteration += 1
        if iteration % 50 == 0:
            print(f"Progress: {iteration}/{total_stat} ({100*iteration/total_stat:.1f}%)")
        
        try:
            # Get method-specific features
            method_features = base_features + [
                col for col in df.columns if col.endswith(prep_method)
            ]
            
            X = df[method_features]
            y = df[target]
            
            # Get score function and estimator
            score_func = stat_score_funcs[score_name]
            estimator = estimators[est_name]
            
            # Evaluate
            results = evaluate_with_statistical_selection(
                X, y, score_func, n_feat, corr_thresh, estimator, cv_splits
            )
            
            # Store results
            row = {
                'stage': stage_name,
                'preprocessing_method': prep_method,
                'selection_method': f'Statistical_{score_name}',
                'base_estimator': est_name,
                'n_features_requested': n_feat,
                'corr_threshold': corr_thresh,
                'cv_folds': cv_splits
            }
            row.update(results)
            all_results.append(row)
            
        except Exception as e:
            print(f"Error: {stage_name}/{prep_method}/{score_name}/{est_name}: {e}")
            continue
    
    # Part 2: Sequential methods
    print("\n" + "="*80)
    print("PART 2: SEQUENTIAL FEATURE SELECTION (with correlation filtering)")
    print("="*80)
    
    seq_param_grid = {
        'stage': [('stage_1', stage_1_df), ('stage_2', stage_2_df)],
        'preprocessing': preprocessing_methods,
        'sequential_method': list(sequential_methods.keys()),
        'estimator': list(estimators.keys()),
        'n_features': n_features_options,
        'corr_threshold': corr_thresholds
    }
    
    total_seq = np.prod([len(v) for v in seq_param_grid.values()])
    print(f"Sequential configurations to test: {total_seq}")
    
    iteration = 0
    
    for (stage_name, df), prep_method, seq_method_name, est_name, n_feat, corr_thresh in product(
        seq_param_grid['stage'],
        seq_param_grid['preprocessing'],
        seq_param_grid['sequential_method'],
        seq_param_grid['estimator'],
        seq_param_grid['n_features'],
        seq_param_grid['corr_threshold']
    ):
        iteration += 1
        if iteration % 20 == 0:
            print(f"Progress: {iteration}/{total_seq} ({100*iteration/total_seq:.1f}%)")

        # Skip incompatible RFE combinations
        if seq_method_name == 'RFE' and est_name in ['SVC', 'KNN', 'MLP', 'MLP_2layer']:
            continue
            
        try:
            # Get method-specific features
            method_features = base_features + [
                col for col in df.columns if col.endswith(prep_method)
            ]
            
            # Skip if not enough features
            if len(method_features) < n_feat:
                continue
            
            X = df[method_features]
            y = df[target]
            
            # Get sequential method config and estimator
            seq_method_config = sequential_methods[seq_method_name]
            estimator = estimators[est_name]
            
            # Evaluate
            results = evaluate_with_sequential_selection(
                X, y, seq_method_name, seq_method_config, 
                estimator, n_feat, corr_thresh, cv_splits
            )
            
            # Store results
            row = {
                'stage': stage_name,
                'preprocessing_method': prep_method,
                'selection_method': seq_method_name,
                'base_estimator': est_name,
                'n_features_requested': n_feat,
                'corr_threshold': corr_thresh,
                'cv_folds': cv_splits
            }
            row.update(results)
            all_results.append(row)
            
        except Exception as e:
            print(f"Error: {stage_name}/{prep_method}/{seq_method_name}/{est_name}: {e}")
            continue
    
    results_df = pd.DataFrame(all_results)
    return results_df

In [3]:
# Base features (always included)
base_features = [
    "hypertension", "early_rupture", "meconium", "induced", "no_progress",
    "is_male", "fetal_weight", "weeks", "is_first_birth", "age",
    "prev_birth", "normal_pres", "stage_1_percent", 
    "stage_1_normalized", "stage_2_normalized"
]

# Signal processing methods
preprocessing_methods = [
    "drop_long_then_autoencoder",
    "drop_long_then_interpolate",
    "drop_long_then_weighted_fill",
    "drop_zeros",
    "interpolate"
]

# Sequential methods
sequential_methods = {
    'SFS_forward': {
        'class': SequentialFeatureSelector,
        'params': {
            'direction': 'forward',
            'n_jobs': -1,
            'cv': 3
        }
    },
    'SFS_backward': {
        'class': SequentialFeatureSelector,
        'params': {
            'direction': 'backward',
            'n_jobs': -1,
            'cv': 3
        }
    },
    'RFE': {
        'class': RFE,
        'params': {
            'step': 1
        }
    }
}

# Base estimators
estimators = {
    'LogisticRegression': LogisticRegression(
        max_iter=1000, 
        class_weight='balanced', 
        random_state=RANDOM_STATE
    ),
    'LogisticRegression_l1': LogisticRegression(
        penalty='l1',
        solver='liblinear',
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=10,
        class_weight='balanced', 
        random_state=RANDOM_STATE,
        max_depth=5
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=10,
        random_state=RANDOM_STATE,
        max_depth=5
    ),
    'SVC': SVC(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        kernel='rbf'
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5
    ),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(5,),
        max_iter=500,
        random_state=RANDOM_STATE,
        early_stopping=True
    ),
    'MLP_2layer': MLPClassifier(
        hidden_layer_sizes=(10,5),
        max_iter=500,
        random_state=RANDOM_STATE,
        early_stopping=True
    )
}

# Number of features to try
n_features_options = [
    4,
    6,
    8,
    10,
    15
]

# Correlation thresholds
corr_thresholds = [0.5, 0.7, 0.9]

# CV splits
cv_splits = 3

# Statistical score functions
stat_score_funcs = {
    'f_classif': f_classif,
    'mutual_info': mutual_info_classif
}

In [4]:
stage_1_df = pd.read_csv("data/stage_1.csv", index_col=0)
stage_2_df = pd.read_csv("data/stage_2.csv", index_col=0)

target = "is_unhealthy"

print(f"Baseline (majority class): {1 - stage_1_df['is_unhealthy'].mean():.4f}")

Baseline (majority class): 0.6491


In [5]:
results_path = "results/classification_results.csv"

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
else: 
    results_df = run_sequential_search(
        base_features,
        preprocessing_methods, 
        sequential_methods, 
        estimators,
        n_features_options,
        corr_thresholds,
        cv_splits,
        stat_score_funcs
    )
    os.makedirs("results", exist_ok=True)
    results_df.to_csv(results_path, index=False)

In [6]:
# Display top configurations
print("\n" + "="*100)
print("TOP 20 CONFIGURATIONS BY ACCURACY")
print("="*100)
top_configs = results_df.nlargest(20, 'accuracy_mean')[
    ['stage', 'preprocessing_method', 'selection_method', 'base_estimator',
        'n_features_requested', 'n_features_selected', 'accuracy_mean', 
        'balanced_accuracy_mean', 'recall_mean', 'specificity_mean', 'balanced_accuracy_std']
]
top_configs


TOP 20 CONFIGURATIONS BY ACCURACY


,stage,preprocessing_method,selection_method,base_estimator,n_features_requested,n_features_selected,accuracy_mean,balanced_accuracy_mean,recall_mean,specificity_mean,balanced_accuracy_std
2878,stage_1,drop_long_then_interpolate,SFS_backward,GradientBoosting,15,15,0.697368,0.614586,0.337607,0.891565,0.020316
3292,stage_1,drop_long_then_weighted_fill,RFE,GradientBoosting,8,8,0.688596,0.610521,0.349478,0.871565,0.052413
3352,stage_1,drop_zeros,SFS_forward,GradientBoosting,8,8,0.688596,0.590510,0.262108,0.918912,0.010723
3353,stage_1,drop_zeros,SFS_forward,GradientBoosting,8,8,0.688596,0.590510,0.262108,0.918912,0.010723
3478,stage_1,drop_zeros,SFS_backward,GradientBoosting,15,15,0.688596,0.608156,0.338082,0.878231,0.024648
5003,stage_2,drop_zeros,SFS_backward,KNN,8,8,0.684211,0.610944,0.363248,0.858639,0.032472
2801,stage_1,drop_long_then_interpolate,SFS_forward,MLP,10,10,0.684211,0.579438,0.226496,0.932381,0.056180
3172,stage_1,drop_long_then_weighted_fill,SFS_backward,GradientBoosting,8,8,0.684211,0.604586,0.337607,0.871565,0.028263
3175,stage_1,drop_long_then_weighted_fill,SFS_backward,GradientBoosting,10,10,0.684211,0.599518,0.313865,0.885170,0.045998
3379,stage_1,drop_zeros,SFS_forward,KNN,6,6,0.684211,0.621520,0.412156,0.830884,0.007581


In [12]:
def get_stats_by(method, sort_by="mean"):
    return results_df.groupby(method).agg({
        'accuracy_mean': ['mean', 'std', 'max'],
        'balanced_accuracy_mean': ['mean', 'std', 'max'],
        'recall_mean': ['mean', 'std', 'max'],
        'specificity_mean': ['mean', 'std', 'max'],
    }).round(4).sort_values(("accuracy_mean", sort_by), ascending=False)

In [13]:
# Summary statistics
print("\n" + "="*100)
print("SUMMARY BY SELECTION METHOD => BEST MEAN ACCURACY")
print("="*100)

selection_method = get_stats_by("selection_method")
selection_method


SUMMARY BY SELECTION METHOD => BEST MEAN ACCURACY


accuracy_mean                 balanced_accuracy_mean  \
                                 mean     std     max                   mean   
selection_method                                                               
RFE                            0.5745  0.0363  0.6886                 0.5342   
SFS_forward                    0.5651  0.0891  0.6886                 0.5175   
SFS_backward                   0.5612  0.0761  0.6974                 0.5285   
Statistical_f_classif          0.5537  0.0740  0.6667                 0.5252   
Statistical_mutual_info        0.5366  0.0802  0.6798                 0.5069   

                                        recall_mean                  \
                            std     max        mean     std     max   
selection_method                                                      
RFE                      0.0322  0.6105      0.3982  0.1290  0.7023   
SFS_forward              0.0374  0.6322      0.3572  0.2607  1.0000   
SFS_backward             0.0339  0.6281      0.4177  0.2366  1.0000   
Statistical_f_classif    0.0396  0.6366      0.4278  0.2524  1.0000   
Statistical_mutual_info  0.0354  0.6180      0.4066  0.2474  1.0000   

                        specificity_mean                  
                                    mean     std     max  
selection_method                                          
RFE                               0.6703  0.1043  0.9388  
SFS_forward                       0.6779  0.2668  1.0000  
SFS_backward                      0.6394  0.2341  1.0000  
Statistical_f_classif             0.6227  0.2354  1.0000  
Statistical_mutual_info           0.6072  0.2459  1.0000

In [14]:
# Summary statistics
print("\n" + "="*100)
print("SUMMARY BY SELECTION METHOD => BEST MAX ACCURACY")
print("="*100)

selection_method = get_stats_by("selection_method", "max")
selection_method


SUMMARY BY SELECTION METHOD => BEST MAX ACCURACY


accuracy_mean                 balanced_accuracy_mean  \
                                 mean     std     max                   mean   
selection_method                                                               
SFS_backward                   0.5612  0.0761  0.6974                 0.5285   
RFE                            0.5745  0.0363  0.6886                 0.5342   
SFS_forward                    0.5651  0.0891  0.6886                 0.5175   
Statistical_mutual_info        0.5366  0.0802  0.6798                 0.5069   
Statistical_f_classif          0.5537  0.0740  0.6667                 0.5252   

                                        recall_mean                  \
                            std     max        mean     std     max   
selection_method                                                      
SFS_backward             0.0339  0.6281      0.4177  0.2366  1.0000   
RFE                      0.0322  0.6105      0.3982  0.1290  0.7023   
SFS_forward              0.0374  0.6322      0.3572  0.2607  1.0000   
Statistical_mutual_info  0.0354  0.6180      0.4066  0.2474  1.0000   
Statistical_f_classif    0.0396  0.6366      0.4278  0.2524  1.0000   

                        specificity_mean                  
                                    mean     std     max  
selection_method                                          
SFS_backward                      0.6394  0.2341  1.0000  
RFE                               0.6703  0.1043  0.9388  
SFS_forward                       0.6779  0.2668  1.0000  
Statistical_mutual_info           0.6072  0.2459  1.0000  
Statistical_f_classif             0.6227  0.2354  1.0000

In [15]:
print("\n" + "="*100)
print("SUMMARY BY BASE ESTIMATOR => BEST MEAN ACCURACY")
print("="*100)

estimator_method = get_stats_by("base_estimator")
estimator_method


SUMMARY BY BASE ESTIMATOR => BEST MEAN ACCURACY


accuracy_mean                 balanced_accuracy_mean  \
                               mean     std     max                   mean   
base_estimator                                                               
GradientBoosting             0.6166  0.0303  0.6974                 0.5263   
KNN                          0.5914  0.0333  0.6842                 0.5123   
RandomForest                 0.5695  0.0300  0.6579                 0.5169   
LogisticRegression           0.5568  0.0381  0.6491                 0.5387   
LogisticRegression_l1        0.5548  0.0480  0.6491                 0.5312   
MLP                          0.5516  0.1108  0.6842                 0.5153   
SVC                          0.5431  0.0466  0.6535                 0.5174   
MLP_2layer                   0.4495  0.1177  0.6491                 0.5042   

                                      recall_mean                  \
                          std     max        mean     std     max   
base_estimator                                                      
GradientBoosting       0.0307  0.6154      0.2229  0.0549  0.4131   
KNN                    0.0379  0.6322      0.2465  0.0673  0.5009   
RandomForest           0.0337  0.6180      0.3385  0.0650  0.5375   
LogisticRegression     0.0397  0.6366      0.4765  0.0919  0.7388   
LogisticRegression_l1  0.0473  0.6206      0.4507  0.1558  0.7887   
MLP                    0.0309  0.6303      0.3932  0.3789  1.0000   
SVC                    0.0386  0.5955      0.4285  0.0995  0.7023   
MLP_2layer             0.0170  0.5824      0.6877  0.3883  1.0000   

                      specificity_mean                  
                                  mean     std     max  
base_estimator                                          
GradientBoosting                0.8297  0.0437  0.9456  
KNN                             0.7782  0.0374  0.8920  
RandomForest                    0.6954  0.0396  0.7977  
LogisticRegression              0.6009  0.0680  0.9256  
LogisticRegression_l1           0.6117  0.1201  1.0000  
MLP                             0.6374  0.3695  1.0000  
SVC                             0.6063  0.0977  0.9392  
MLP_2layer                      0.3207  0.3895  1.0000

In [16]:
print("\n" + "="*100)
print("SUMMARY BY BASE ESTIMATOR => BEST MAX ACCURACY")
print("="*100)

estimator_method = get_stats_by("base_estimator", "max")
estimator_method


SUMMARY BY BASE ESTIMATOR => BEST MAX ACCURACY


accuracy_mean                 balanced_accuracy_mean  \
                               mean     std     max                   mean   
base_estimator                                                               
GradientBoosting             0.6166  0.0303  0.6974                 0.5263   
KNN                          0.5914  0.0333  0.6842                 0.5123   
MLP                          0.5516  0.1108  0.6842                 0.5153   
RandomForest                 0.5695  0.0300  0.6579                 0.5169   
SVC                          0.5431  0.0466  0.6535                 0.5174   
LogisticRegression           0.5568  0.0381  0.6491                 0.5387   
LogisticRegression_l1        0.5548  0.0480  0.6491                 0.5312   
MLP_2layer                   0.4495  0.1177  0.6491                 0.5042   

                                      recall_mean                  \
                          std     max        mean     std     max   
base_estimator                                                      
GradientBoosting       0.0307  0.6154      0.2229  0.0549  0.4131   
KNN                    0.0379  0.6322      0.2465  0.0673  0.5009   
MLP                    0.0309  0.6303      0.3932  0.3789  1.0000   
RandomForest           0.0337  0.6180      0.3385  0.0650  0.5375   
SVC                    0.0386  0.5955      0.4285  0.0995  0.7023   
LogisticRegression     0.0397  0.6366      0.4765  0.0919  0.7388   
LogisticRegression_l1  0.0473  0.6206      0.4507  0.1558  0.7887   
MLP_2layer             0.0170  0.5824      0.6877  0.3883  1.0000   

                      specificity_mean                  
                                  mean     std     max  
base_estimator                                          
GradientBoosting                0.8297  0.0437  0.9456  
KNN                             0.7782  0.0374  0.8920  
MLP                             0.6374  0.3695  1.0000  
RandomForest                    0.6954  0.0396  0.7977  
SVC                             0.6063  0.0977  0.9392  
LogisticRegression              0.6009  0.0680  0.9256  
LogisticRegression_l1           0.6117  0.1201  1.0000  
MLP_2layer                      0.3207  0.3895  1.0000

In [17]:
print("\n" + "="*100)
print("SUMMARY BY PROCESSING METHOD => BEST MEAN ACCURACY")
print("="*100)

processing_method = get_stats_by("preprocessing_method")
processing_method


SUMMARY BY PROCESSING METHOD => BEST MEAN ACCURACY


accuracy_mean                  \
                                      mean     std     max   
preprocessing_method                                         
drop_long_then_weighted_fill        0.5581  0.0775  0.6886   
drop_long_then_interpolate          0.5569  0.0762  0.6974   
drop_zeros                          0.5562  0.0774  0.6886   
drop_long_then_autoencoder          0.5559  0.0753  0.6798   
interpolate                         0.5549  0.0806  0.6842   

                             balanced_accuracy_mean                  \
                                               mean     std     max   
preprocessing_method                                                  
drop_long_then_weighted_fill                 0.5228  0.0365  0.6322   
drop_long_then_interpolate                   0.5211  0.0361  0.6299   
drop_zeros                                   0.5207  0.0382  0.6299   
drop_long_then_autoencoder                   0.5189  0.0342  0.6366   
interpolate                                  0.5224  0.0410  0.6281   

                             recall_mean              specificity_mean  \
                                    mean     std  max             mean   
preprocessing_method                                                     
drop_long_then_weighted_fill      0.4031  0.2360  1.0           0.6425   
drop_long_then_interpolate        0.3996  0.2397  1.0           0.6425   
drop_zeros                        0.4005  0.2422  1.0           0.6409   
drop_long_then_autoencoder        0.3936  0.2416  1.0           0.6442   
interpolate                       0.4125  0.2422  1.0           0.6324   

                                           
                                 std  max  
preprocessing_method                       
drop_long_then_weighted_fill  0.2343  1.0  
drop_long_then_interpolate    0.2343  1.0  
drop_zeros                    0.2362  1.0  
drop_long_then_autoencoder    0.2352  1.0  
interpolate                   0.2395  1.0

In [21]:
print("\n" + "="*100)
print("SUMMARY BY PROCESSING METHOD => BEST MAX ACCURACY")
print("="*100)

processing_method = get_stats_by("preprocessing_method", "max")
processing_method


SUMMARY BY PROCESSING METHOD => BEST MAX ACCURACY


accuracy_mean                  \
                                      mean     std     max   
preprocessing_method                                         
drop_long_then_interpolate          0.5569  0.0762  0.6974   
drop_zeros                          0.5562  0.0774  0.6886   
drop_long_then_weighted_fill        0.5581  0.0775  0.6886   
interpolate                         0.5549  0.0806  0.6842   
drop_long_then_autoencoder          0.5559  0.0753  0.6798   

                             balanced_accuracy_mean                  \
                                               mean     std     max   
preprocessing_method                                                  
drop_long_then_interpolate                   0.5211  0.0361  0.6299   
drop_zeros                                   0.5207  0.0382  0.6299   
drop_long_then_weighted_fill                 0.5228  0.0365  0.6322   
interpolate                                  0.5224  0.0410  0.6281   
drop_long_then_autoencoder                   0.5189  0.0342  0.6366   

                             recall_mean              specificity_mean  \
                                    mean     std  max             mean   
preprocessing_method                                                     
drop_long_then_interpolate        0.3996  0.2397  1.0           0.6425   
drop_zeros                        0.4005  0.2422  1.0           0.6409   
drop_long_then_weighted_fill      0.4031  0.2360  1.0           0.6425   
interpolate                       0.4125  0.2422  1.0           0.6324   
drop_long_then_autoencoder        0.3936  0.2416  1.0           0.6442   

                                           
                                 std  max  
preprocessing_method                       
drop_long_then_interpolate    0.2343  1.0  
drop_zeros                    0.2362  1.0  
drop_long_then_weighted_fill  0.2343  1.0  
interpolate                   0.2395  1.0  
drop_long_then_autoencoder    0.2352  1.0

In [19]:
print("\n" + "="*100)
print("SUMMARY BY STAGE => BEST MEAN ACCURACY")
print("="*100)

processing_method = get_stats_by("stage")
processing_method


SUMMARY BY STAGE => BEST MEAN ACCURACY


accuracy_mean                 balanced_accuracy_mean                  \
                 mean     std     max                   mean     std     max   
stage                                                                          
stage_1        0.5564  0.0787  0.6974                 0.5219  0.0364  0.6366   
stage_2        0.5564  0.0761  0.6842                 0.5204  0.0381  0.6322   

        recall_mean              specificity_mean               
               mean     std  max             mean     std  max  
stage                                                           
stage_1      0.4055  0.2456  1.0           0.6383  0.2415  1.0  
stage_2      0.3982  0.2350  1.0           0.6427  0.2302  1.0

In [20]:
print("\n" + "="*100)
print("SUMMARY BY STAGE => BEST MAX ACCURACY")
print("="*100)

processing_method = get_stats_by("stage", "max")
processing_method


SUMMARY BY STAGE => BEST MAX ACCURACY


accuracy_mean                 balanced_accuracy_mean                  \
                 mean     std     max                   mean     std     max   
stage                                                                          
stage_1        0.5564  0.0787  0.6974                 0.5219  0.0364  0.6366   
stage_2        0.5564  0.0761  0.6842                 0.5204  0.0381  0.6322   

        recall_mean              specificity_mean               
               mean     std  max             mean     std  max  
stage                                                           
stage_1      0.4055  0.2456  1.0           0.6383  0.2415  1.0  
stage_2      0.3982  0.2350  1.0           0.6427  0.2302  1.0